# Multi-Head Latent Attention with RoPE

Notebook 04 built MLA without positional information. Real models need **position
awareness**, and DeepSeek uses **rotary position embedding (RoPE)**. The catch: RoPE
rotates queries and keys by an amount that depends on their position, so it cannot be
folded into the compressed latent the way plain MLA content keys can.

DeepSeek solves this with **decoupled RoPE**. Each head splits its query and key into two
parts:

| Part | Source | RoPE applied | Cached as |
| - | - | - | - |
| content (nope) | the compressed latent `c_kv` | no | the latent `c_kv` |
| rotary (rope) | a small separate projection | yes | one shared rope key |

The attention score is the sum of the two parts:

$$
\text{score}_{ij} = \frac{q^{C}_i \cdot k^{C}_j \;+\; q^{R}_i \cdot k^{R}_j}{\sqrt{d_{head} + d_{rope}}}
$$

The content keys stay compressed inside the latent, and only a small rotary key (shared
across heads) is added, so the KV cache stays small while the model still knows where each
token sits in the sequence.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Rotary position embedding

RoPE encodes a position by **rotating** pairs of dimensions by an angle proportional to
the position. Two tokens that are the same distance apart always share the same relative
rotation, which is how the model reads relative position.

- `build_cos_sin` precomputes the rotation angles for each position.
- `rotate_half` swaps and negates the two halves of a vector.
- `apply_rope` applies the rotation `x * cos + rotate_half(x) * sin`.

In [2]:
def build_cos_sin(positions, dim, base=10000.0, device="cpu"):
    # Rotation angles for each position, shape (S, dim)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    freqs = positions[:, None].float() * inv_freq[None, :]   # (S, dim/2)
    emb = torch.cat([freqs, freqs], dim=-1)                  # (S, dim)
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    # x: (B, S, H, dim); cos/sin: (S, dim) broadcast over batch and heads
    cos = cos[None, :, None, :]
    sin = sin[None, :, None, :]
    return x * cos + rotate_half(x) * sin

## The MLA layer

Same structure as `RopelessMLA` from notebook 04, with two extra projections for the
rotary path.

| Layer | Maps | Purpose |
| - | - | - |
| `W_dkv` | `d_model -> kv_latent_dim` | compress a token into the KV latent |
| `W_uk` | `kv_latent_dim -> d_model` | rebuild content keys |
| `W_uv` | `kv_latent_dim -> d_model` | rebuild values |
| `W_q` | `d_model -> d_model` | content queries |
| `W_qr` | `d_model -> n_heads * rope_dim` | rotary queries (one per head) |
| `W_kr` | `d_model -> rope_dim` | rotary key (shared across heads) |
| `W_o` | `d_model -> d_model` | output projection |

`forward` returns the output and a cache tuple `(c_kv, k_rope)`: the compressed latent plus
the small shared rotary key.

In [3]:
class MLA(nn.Module):
    """Multi-Head Latent Attention with decoupled RoPE (the DeepSeek design).

    Keys and values are compressed into a small latent `c_kv` (cached), exactly like
    RopelessMLA. Position information is added through a separate rotary path: a small
    per-head rotary query and one rotary key shared across heads, both rotated by RoPE.
    """
    def __init__(self, d_model, n_heads, kv_latent_dim, rope_dim, base=10000.0):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads      # content dim per head
        self.rope_dim = rope_dim                # rotary dim per head / shared key
        self.base = base

        # Content path (same idea as RopelessMLA)
        self.W_q   = nn.Linear(d_model, n_heads * self.head_dim, bias=False)  # content queries
        self.W_dkv = nn.Linear(d_model, kv_latent_dim, bias=False)            # compress K/V
        self.W_uk  = nn.Linear(kv_latent_dim, d_model, bias=False)            # rebuild keys
        self.W_uv  = nn.Linear(kv_latent_dim, d_model, bias=False)            # rebuild values

        # Rotary path (decoupled RoPE)
        self.W_qr  = nn.Linear(d_model, n_heads * rope_dim, bias=False)       # rotary queries (per head)
        self.W_kr  = nn.Linear(d_model, rope_dim, bias=False)                 # rotary key (shared)

        self.W_o = nn.Linear(d_model, d_model, bias=False)                    # output projection
        self.ln = nn.LayerNorm(kv_latent_dim)

    def forward(self, x, kv_cache=None, past_length=0):
        B, S, D = x.size()
        H, hd, rd = self.n_heads, self.head_dim, self.rope_dim

        # RoPE angles for the absolute positions of these tokens
        positions = torch.arange(past_length, past_length + S, device=x.device)
        cos, sin = build_cos_sin(positions, rd, self.base, x.device)

        # Compress this step's tokens and build its shared rotary key
        new_c_kv = self.ln(self.W_dkv(x))                                     # (B, S, latent)
        new_k_rope = apply_rope(self.W_kr(x).view(B, S, 1, rd), cos, sin).view(B, S, rd)

        # Append to the cache
        if kv_cache is None:
            c_kv, k_rope = new_c_kv, new_k_rope
        else:
            c_kv = torch.cat([kv_cache[0], new_c_kv], dim=1)                  # (B, S_full, latent)
            k_rope = torch.cat([kv_cache[1], new_k_rope], dim=1)             # (B, S_full, rope_dim)
        S_full = c_kv.size(1)

        # Decompress content keys and values, split into heads
        k_c = self.W_uk(c_kv).view(B, S_full, H, hd).transpose(1, 2)          # (B, H, S_full, hd)
        v   = self.W_uv(c_kv).view(B, S_full, H, hd).transpose(1, 2)          # (B, H, S_full, hd)

        # Queries: content part plus rotary part
        q_c = self.W_q(x).view(B, S, H, hd).transpose(1, 2)                   # (B, H, S, hd)
        q_r = apply_rope(self.W_qr(x).view(B, S, H, rd), cos, sin).transpose(1, 2)  # (B, H, S, rd)

        # Scores = content score + rotary score, then scale
        content = q_c @ k_c.transpose(-2, -1)                                 # (B, H, S, S_full)
        rope = q_r @ k_rope.unsqueeze(1).transpose(-2, -1)                    # (B, H, S, S_full)
        scores = (content + rope) / (hd + rd) ** 0.5

        # Causal mask, softmax, weighted values, merge heads
        mask = torch.tril(torch.ones(S, S_full, device=x.device), diagonal=past_length)
        scores = scores.masked_fill(mask[None, None] == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, S, D)        # (B, S, D)

        return self.W_o(out), (c_kv, k_rope)

## Running the layer

A forward pass on one sequence of six tokens. The output keeps the `d_model` width, while
the cache holds only the latent and the shared rotary key.

In [4]:
torch.manual_seed(0)
mla = MLA(d_model=8, n_heads=2, kv_latent_dim=4, rope_dim=4).eval()

x = torch.randn(1, 6, 8)          # (batch=1, seq_len=6, d_model=8)
with torch.no_grad():
    output, (c_kv, k_rope) = mla(x)

print("input shape        :", x.shape)
print("output shape       :", output.shape)         # (1, 6, 8)
print("latent cache shape :", c_kv.shape)           # (1, 6, 4)
print("rope-key cache     :", k_rope.shape)         # (1, 6, 4)  shared across heads

input shape        : torch.Size([1, 6, 8])
output shape       : torch.Size([1, 6, 8])
latent cache shape : torch.Size([1, 6, 4])
rope-key cache     : torch.Size([1, 6, 4])


## The payoff: small cache and position awareness

Three checks. First, generating token by token while caching only `(c_kv, k_rope)` must
match the full forward pass. Second, because of RoPE, reordering the tokens must change the
output, which confirms the layer is position aware. Third, we compare the cache size with a
standard key and value cache.

In [5]:
# 1) Incremental decoding, caching only (c_kv, k_rope), must match the full forward pass
torch.manual_seed(0)
with torch.no_grad():
    cache, outputs = None, []
    for t in range(x.size(1)):
        step_out, cache = mla(x[:, t:t+1, :], kv_cache=cache, past_length=t)
        outputs.append(step_out)
    incremental = torch.cat(outputs, dim=1)
print("max difference vs full forward:", (output - incremental).abs().max().item())

# 2) RoPE makes the layer position aware: reordering tokens changes the output
with torch.no_grad():
    swapped = mla(x[:, torch.tensor([1, 0, 2, 3, 4, 5])])[0]
print("output changes when first two tokens are swapped:",
      not torch.allclose(swapped, output, atol=1e-5))

# 3) Cache size: latent + shared rope key vs a standard key/value cache
d_model, kv_latent_dim, rope_dim = 8, 4, 4
standard  = 2 * d_model                 # standard attention caches full K and V
mla_total = kv_latent_dim + rope_dim    # MLA caches the latent plus one shared rope key
print(f"standard KV cache : {standard} floats / token")
print(f"MLA cache         : {mla_total} floats / token")
print(f"compression       : {standard / mla_total:.1f}x smaller")

max difference vs full forward: 7.450580596923828e-08
output changes when first two tokens are swapped: True
standard KV cache : 16 floats / token
MLA cache         : 8 floats / token
compression       : 2.0x smaller
